**Imports and setup**, including `utils` (spatial smoothing) and the hsiViewer.

In [ ]:
from sklearn import linear_model
import matplotlib.pyplot as plt
from matplotlib import colors
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
import numpy as np
from sklearn.mixture import GaussianMixture
import numpy as np
import copy
import spectral
import time
import csv
import os
import importlib
import pickle
from upwins_hsi import utils
from hsiViewer import hsi_viewer_layers as hlv
from hsiViewer import hsi_viewer_ROI as hvr
import matplotlib as mpl
mpl.rcParams['lines.linewidth'] = 0.75

# --- Load configuration (paths + parameters live in config.yaml) ---
# config.yaml and the paths inside it are relative to the repo root, but this
# notebook lives in notebooks/. Walk up to the repo root and resolve every
# configured path against it, so this works whether Jupyter is launched from
# the repo root or from notebooks/.
import yaml
from pathlib import Path
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'config.yaml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
with open(REPO_ROOT / 'config.yaml') as _f:
    CONFIG = yaml.safe_load(_f)
for _section in ('paths',):
    for _key, _val in CONFIG.get(_section, {}).items():
        if isinstance(_val, str):
            CONFIG[_section][_key] = str(REPO_ROOT / _val)

**Load the calibration** (`gain`, `offset`) produced by notebook 01.

In [ ]:
# Read gain/offset for this collection. Prefer notebook 01's freshly-written
# bundle in calibration_dir; fall back to the shipped seed set when it is not
# there yet (e.g. a fresh clone), and print which was used so a stale or
# wrong-collection calibration is noticed rather than silently applied.
import os, warnings
_cal_dir  = CONFIG['paths']['calibration_dir']
_seed_dir = CONFIG['paths']['calibration_seed_dir']
_gain   = os.path.join(_cal_dir, 'gain.npy')
_offset = os.path.join(_cal_dir, 'offset.npy')
if not (os.path.exists(_gain) and os.path.exists(_offset)):
    warnings.warn(
        'No gain/offset in calibration_dir (' + _cal_dir + '); falling back to '
        'the shipped seed set in ' + _seed_dir + '. Run notebook 01 for this '
        'collection if it needs its own calibration.')
    _gain   = os.path.join(_seed_dir, 'gain.npy')
    _offset = os.path.join(_seed_dir, 'offset.npy')
gain = np.load(_gain)
offset = np.load(_offset)
print('Using gain:   ' + _gain)
print('Using offset: ' + _offset)

## Open the Image to Convert to Reflectance

**Open a raw image to convert**, and read the smoothing level and bad-band ranges from the config. Set the image in `config.yaml`.

In [ ]:
# number of smoothing iterations
smoothing_level = CONFIG['reflectance']['smoothing_level']
# wl range(s) to remove (in nanometers)
bbl_wl_ranges = CONFIG['reflectance']['bbl_wl_ranges']

# raw image to convert to reflectance (set both entries in config.yaml)
raw_image_hdr = CONFIG['paths']['raw_image_hdr']
raw_image = CONFIG['paths']['raw_image']
assert Path(raw_image_hdr).stem == Path(raw_image).stem, \
    "raw_image and raw_image_hdr name different cubes — check config.yaml"
# Read the image
im = spectral.envi.open(raw_image_hdr, raw_image)
#im.Arr = im.load().astype(np.float32)
#im.mask = im.Arr[:,:,0]!=0
im.wl = np.asarray(im.bands.centers)
wl = im.wl

## Convert Image to Reflectance

**Convert to reflectance.** Drops bad bands, applies per-band gain/offset, masks empty pixels, spatially smooths, and saves the reflectance image next to the raw one.

In [ ]:
# ====== Convert to Reflectance ======
# wl range(s) to remove (in nanometers)
bbl_wl_ranges = CONFIG['reflectance']['bbl_wl_ranges']
indices = []
for i in range(len(wl)):
    is_bad_band = False
    for bbl_wl_range in bbl_wl_ranges:
        if bbl_wl_range[0] < wl[i] < bbl_wl_range[1]:
            is_bad_band = True                
    if (not is_bad_band):
        indices.append(int(i))
indices = np.asarray(indices, dtype=np.int16)

# determine the parameters for the image
nr = im.nrows
nc = im.ncols
nb = len(indices)

# subset the gain and offset to the good bands.
# NOTE: this rebinds gain/offset in place. Re-running this cell without first
# re-running the "Load the calibration" cell double-subsets and raises
# IndexError — re-run that cell first if you need to re-run this one.
gain = gain[indices]
offset = offset[indices]
wl = wl[indices]

# Prepare an output array for the reflectance image
imRef = np.zeros((nr, nc, nb), dtype=np.float32)
# Create the data mask
mask = (im.read_band(0) > 0).astype(np.float32)
    
# ====== Load the image and compute reflectance ======
# Loop over bands and fill the result
print('Reading the image and converting to reflectance.')
for i, b in enumerate(indices):
    imRef[:, :, i] = (gain[i]*np.squeeze(im.read_band(b) + offset[i])*mask).astype(np.float32)
                            
# ====== Spatially smooth the image ======
for i in range(smoothing_level):
    print(f'Smoothing the image, iteration {i+1}.')
    imRef = utils.spatial_smoothing(imRef, mask=mask).astype(np.float32)

# ====== Save the image ======
# Save the image
print('Saving the image.')
md=im.metadata
md['wavelength'] = [str(w) for w in wl]
# Write next to the raw image as <raw_image>_ref.img/.hdr, derived straight from
# the raw_image config key (the single source of truth for this collection).
# save_image derives the .img name from the .hdr path; notebook 02's viewer cell
# below and notebook 03's default both open this same <raw_image>_ref product.
spectral.envi.save_image(CONFIG['paths']['raw_image'] + '_ref.hdr', imRef, metadata=md, force=True)

**Optional — open a reflectance image** to inspect it.

In [ ]:
# Reflectance image to view: notebook 02 always views its OWN output -- the _ref
# product the save cell just wrote next to raw_image. Derive that path from
# raw_image (the single source of truth) instead of re-reading a separate config
# key, so this cell can never open a different cube than was just written. (The
# reflectance_image keys in config.yaml are notebook 03's input, not used here.)
reflectance_image_hdr = CONFIG['paths']['raw_image'] + '_ref.hdr'
reflectance_image     = CONFIG['paths']['raw_image'] + '_ref.img'
# Read the image
im = spectral.envi.open(reflectance_image_hdr, reflectance_image)
im.Arr = im.load().astype(np.float32)
im.mask = im.Arr[:,:,0]!=0
im.wl = np.asarray(im.bands.centers)
wl = im.wl

**Interactive.** Opens the reflectance image in the hsiViewer to examine pixels and spectra.

In [ ]:
# If you want to manually examine the image and sepctra
hlv.viewer(im)